# Tutorial: Spatial site selection

This tutorial demonstrates four fast ways to select STREAM monitoring sites: bounding box, search radius, nearest sites, and Hydrologic Unit Code (HUC). These methods filter the small site metadata and HUC lookup tables; they do not download the large time-series datasets.

## 1. Connect to HydroShare

In [ ]:
import numpy as np

import streamwq as stream
from streamwq.gage_metadata_map import plot_gage_metadata_map


def add_distance_to_center(metadata, center):
    result = metadata.copy()
    center_lon, center_lat = np.radians(center)
    site_lon = np.radians(result["longitude_wgs84"].astype(float))
    site_lat = np.radians(result["latitude_wgs84"].astype(float))
    delta_lon = site_lon - center_lon
    delta_lat = site_lat - center_lat
    haversine = (
        np.sin(delta_lat / 2) ** 2
        + np.cos(center_lat) * np.cos(site_lat) * np.sin(delta_lon / 2) ** 2
    )
    result["distance_km"] = 2 * 6371.0088 * np.arcsin(np.sqrt(np.clip(haversine, 0, 1)))
    return result.sort_values("distance_km").reset_index(drop=True)

## 2. Select sites within a bounding box

A bounding box is written as `(west, south, east, north)` using longitude and latitude in WGS 84. This example covers part of the Mid-Atlantic region and also requires water-temperature observations.

In [ ]:
bbox_sites = stream.get_sites(
    variables="WTemp_C",
    bbox=(-80.0, 36.0, -75.0, 40.0),
)

print(f"Found {len(bbox_sites):,} sites inside the bounding box")
bbox_metadata = stream.get_metadata(site_ids=bbox_sites).select(
    ["STREAM_ID", "site_name", "state_name", "latitude_wgs84", "longitude_wgs84"]
)
bbox_metadata.slice(0, 10)

## 3. Select sites within a radius

The center is ordered as `(longitude, latitude)`. A radius or nearest-count filter returns matching site IDs; fetch their metadata separately with `get_metadata()`. Nearest results are ordered from closest to farthest.

In [ ]:
richmond = (-77.4360, 37.5407)

radius_sites = stream.get_sites(
    variables="WTemp_C",
    center=richmond,
    radius_km=100,
)

print(f"Found {len(radius_sites):,} sites within 100 km")
radius_metadata = stream.get_metadata(
    site_ids=radius_sites,
    columns=["STREAM_ID", "site_name", "latitude_wgs84", "longitude_wgs84"],
).to_pandas()
add_distance_to_center(radius_metadata, richmond)[
    ["STREAM_ID", "site_name", "distance_km"]
].head(10)

## 4. Find the nearest sites

Use `nearest` when you need a fixed number of sites rather than every site inside a fixed distance. Existing filters are applied before distance ranking.

In [ ]:
nearest_sites = stream.get_sites(
    variables=["WTemp_C", "DO_mgL"],
    center=richmond,
    nearest=5,
)

nearest_metadata = stream.get_metadata(
    site_ids=nearest_sites,
    columns=["STREAM_ID", "site_name", "latitude_wgs84", "longitude_wgs84"],
).to_pandas()
add_distance_to_center(nearest_metadata, richmond)[
    ["STREAM_ID", "site_name", "distance_km"]
]

You can combine `radius_km` and `nearest` to request at most five matching sites within 200 km.

In [ ]:
nearby_sites = stream.get_sites(
    center=richmond,
    radius_km=200,
    nearest=5,
)

nearby_metadata = stream.get_metadata(
    site_ids=nearby_sites,
    columns=["STREAM_ID", "site_name", "latitude_wgs84", "longitude_wgs84"],
).to_pandas()
add_distance_to_center(nearby_metadata, richmond)[
    ["STREAM_ID", "site_name", "distance_km"]
]

## 5. Select sites by HUC6

HUC codes should be strings so leading zeros are preserved. HUC membership comes from the small `gage_huc_lookup.parquet` file in the STREAM HydroShare resource, so this operation does not query NHDPlus.

In [ ]:
huc6_sites = stream.get_sites(
    huc6="170900",
    variables="WTemp_C",
)

print(f"Found {len(huc6_sites):,} matching sites in HUC6 170900")
huc6_metadata = stream.get_metadata(
    site_ids=huc6_sites,
    columns=["STREAM_ID", "site_name", "state_name"],
)
huc6_metadata.slice(0, 10)

## 6. Refine the selection to a HUC8

The HUC lookup is exposed through discovery filters, while `get_sites()` returns only STREAM_ID values. This example uses a known HUC8 code; the metadata table can then be fetched separately with `get_metadata()`.

In [ ]:
# 17090001 is a HUC8 in the HUC6 region used above.
example_huc8 = "17090001"
huc8_sites = stream.get_sites(
    huc8=example_huc8,
    variables="WTemp_C",
)

print(f"Found {len(huc8_sites):,} matching sites in HUC8 {example_huc8}")
huc8_metadata = stream.get_metadata(site_ids=huc8_sites).to_pandas()
huc8_metadata[["STREAM_ID", "site_name", "state_name"]]

## 7. Display the selected stations

The lightweight map below displays only the selected station points. Click a point to inspect its metadata.

In [ ]:
plot_gage_metadata_map(huc8_metadata)

## 8. Use the spatial selection to retrieve observations

A discovery result is a one-column Arrow table containing `STREAM_ID`; pass it directly to `get_data()`. Keep the first three sites and a short period for this example.

In [ ]:
selected_sites = huc8_sites.slice(0, 3)
water_quality = stream.get_data(
    "water_quality",
    site_ids=selected_sites,
    columns=["STREAM_ID", "DateTime", "WTemp_C"],
    start="2020-01-01",
    end="2020-01-31 23:59:59",
    quality="approved",
).to_pandas()

print(f"Retrieved {len(water_quality):,} observations")
if water_quality.empty:
    print("No approved observations matched these candidate sites and dates.")
else:
    water_quality.head()

## Next steps

These selectors can be combined to build a study population before reading time-series data. A later tutorial can add clickable HUC boundaries after simplified boundary geometries are made available in HydroShare.